# {Project Title}📝

![Banner](./assets/banner.jpeg)

## Topic
*What problem are you (or your stakeholder) trying to address?*
📝 <!-- Answer Below -->
I want to further understand what factors truly relate to the popularity of video games. Especailly with GTA 6 just around the corner, having garnered huge amounts of attention in every nook and crany of the internet before release. So im interested in what factors can affect a games popularity and whether or not things like reviews, price, and genre strongly influence that.

## Project Question
*What specific question are you seeking to answer with this project?*
*This is not the same as the questions you ask to limit the scope of the project.*
📝 <!-- Answer Below -->
What factors are associated with the populartiy of video games and how can these factors help explain the popularity of GTA 6?

## What would an answer look like?
*What is your hypothesized answer to your question?*
📝 <!-- Answer Below -->
I think gmaes with higher player engagment strong review and certain genres to have generallly higher levels of popularity. I also expect games of established franchise to have higher popularity than less establised franchises. For GTA 6 comparing it to games of similar stature could lead to some promising results. The analysis would use charts such as scatter plots and bar charts, along with statistics such as correlations and median ownership.

## Data Sources
*What 3 data sources have you identified for this project?*
*How are you going to relate these datasets?*
📝 <!-- Answer Below -->
Steam Games Dataset (games.csv) – Contains information about games including estimated owners, price, reviews, Metacritic scores, peak concurrent players, playtime, genres, and release dates.
Backloggd Games Dataset (backloggd_games.csv) – Contains information about games including ratings, platforms, genres, plays, backlogs, wishlists, and reviews.
Video Games Dataset (video_games.csv) – Contains information about games including different play styles, playtime, and console.

## Approach and Analysis
*What is your approach to answering your project question?*
*How will you use the identified data to answer your project question?*
📝 <!-- Start Discussing the project here; you can add as many code cells as you need -->

In [1]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
CACHE_DIR = Path("api_cache")
CACHE_DIR.mkdir(exist_ok=True)
SAMPLE_SIZE = 300
HEADERS = {"User-Agent": "VideoGamePopularityCourseProject/1.0 (academic data analysis)"}
session = requests.Session()
session.headers.update(HEADERS)
retrieved_at_utc = datetime.now(timezone.utc).isoformat()

In [2]:
def api_json(url, params, cache_name, pause=0.0):
    path = CACHE_DIR / cache_name
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    for attempt in range(3):
        try:
            response = session.get(url, params=params, timeout=25)
            if response.status_code == 429 or 500 <= response.status_code < 600:
                wait = min(int(response.headers.get("Retry-After", 2 ** (attempt + 1))), 60)
                time.sleep(wait)
                continue
            response.raise_for_status()
            payload = response.json()
            path.write_text(json.dumps(payload), encoding="utf-8")
            if pause:
                time.sleep(pause)
            return payload
        except (requests.RequestException, ValueError) as exc:
            if attempt == 2:
                print(f"Unable to retrieve {cache_name}: {exc}")
                return None
            time.sleep(2 ** attempt)
    print(f"Skipped {cache_name}: request remained unavailable")
    return None

steamspy_api = "https://steamspy.com/api.php"
steam_store_api = "https://store.steampowered.com/api/appdetails"
cheapshark_api = "https://www.cheapshark.com/api/1.0/games"

In [3]:
recent = api_json(steamspy_api, {"request": "top100in2weeks"}, "steamspy_top2weeks.json", pause=1.1)
all_time = api_json(steamspy_api, {"request": "top100forever"}, "steamspy_top_forever.json", pause=1.1)
if recent is None and all_time is None:
    raise RuntimeError("SteamSpy is unavailable and there is no cached snapshot. Run with internet access.")

def ranked_rows(payload):
    if isinstance(payload, dict):
        return [r for r in payload.values() if isinstance(r, dict) and r.get("appid")]
    if isinstance(payload, list):
        return [r for r in payload if isinstance(r, dict) and r.get("appid")]
    return []

recent_rows = ranked_rows(recent)
all_time_rows = ranked_rows(all_time)
selected = {}
for candidate in (recent_rows[:SAMPLE_SIZE // 2] + all_time_rows[:SAMPLE_SIZE] + recent_rows):
    try:
        appid = int(candidate["appid"])
    except (ValueError, TypeError):
        continue
    selected.setdefault(appid, candidate)
    if len(selected) >= SAMPLE_SIZE:
        break

steamspy_raw = pd.DataFrame(list(selected.values()))
if steamspy_raw.empty:
    raise RuntimeError("No valid SteamSpy game records were returned.")
steamspy_raw["appid"] = pd.to_numeric(steamspy_raw["appid"], errors="coerce").astype("Int64")
print("SteamSpy selected rows:", len(steamspy_raw))
display(steamspy_raw.head(10))

SteamSpy selected rows: 100


,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
0,730,Counter-Strike: Global Offensive,Valve,Valve,,7642084,1173003,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,1013936
1,1172470,Apex Legends,Respawn,Electronic Arts,,668053,326926,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,124262
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",,1520457,1037487,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,314682
3,1623730,Palworld,Pocketpair,Pocketpair,,358266,22443,0,"50,000,000 .. 100,000,000",0,0,0,0,2999,2999,0,18028
4,440,Team Fortress 2,Valve,Valve,,1044264,117208,0,"50,000,000 .. 100,000,000",0,0,0,0,0,0,0,43819
5,1938090,Call of Duty: Modern Warfare II,"Treyarch, Raven Software, Beenox, High Moon St...",Activision,,419594,294520,0,"50,000,000 .. 100,000,000",0,0,0,0,3849,6999,45,67419
6,1063730,New World: Aeternum,Amazon Games,Amazon Games,,196798,90080,0,"50,000,000 .. 100,000,000",0,0,0,0,5999,5999,0,5583
7,2358720,Black Myth: Wukong,Game Science,Game Science,,1111720,38378,0,"50,000,000 .. 100,000,000",0,0,0,0,5999,5999,0,15004
8,271590,Grand Theft Auto V Legacy,Rockstar North,Rockstar Games,,1739980,250576,0,"50,000,000 .. 100,000,000",0,0,0,0,0,0,0,67851
9,550,Left 4 Dead 2,Valve,Valve,,940221,23762,0,"50,000,000 .. 100,000,000",0,0,0,0,999,999,0,25122


In [5]:
store_records = []
cheap_records = []
for appid in steamspy_raw["appid"].dropna().astype(int):
    store = api_json(steam_store_api, {"appids": appid, "cc": "us", "l": "english"}, f"steam_store_{appid}.json", pause=0.4)
    root = (store or {}).get(str(appid), {}) if isinstance(store, dict) else {}
    data = root.get("data", {}) if root.get("success") else {}
    if data.get("type") == "game":
        price = data.get("price_overview") or {}
        store_records.append({
            "appid": appid,
            "store_name": data.get("name"),
            "release_date_raw": (data.get("release_date") or {}).get("date"),
            "coming_soon": (data.get("release_date") or {}).get("coming_soon"),
            "genre_primary": ((data.get("genres") or [{}])[0]).get("description"),
            "store_price_usd": price.get("final", np.nan) / 100 if isinstance(price.get("final"), (int, float)) else (0.0 if data.get("is_free") is True else np.nan),
            "store_list_price_usd": price.get("initial", np.nan) / 100 if isinstance(price.get("initial"), (int, float)) else (0.0 if data.get("is_free") is True else np.nan),
            "metacritic_score": (data.get("metacritic") or {}).get("score"),
        })
    cheap = api_json(cheapshark_api, {"steamAppID": appid}, f"cheapshark_{appid}.json", pause=0.4)
    if isinstance(cheap, list):
        matches = [x for x in cheap if str(x.get("steamAppID")) == str(appid)]
        offers = pd.to_numeric(pd.Series([x.get("cheapest") for x in matches], dtype="object"), errors="coerce")
        cheap_records.append({"appid": appid, "lowest_other_store_usd": offers.min() if not offers.dropna().empty else np.nan, "cheapshark_matches": len(matches)})

store_raw = pd.DataFrame(store_records, columns=["appid", "store_name", "release_date_raw", "coming_soon", "genre_primary", "store_price_usd", "store_list_price_usd", "metacritic_score"])
cheap_raw = pd.DataFrame(cheap_records, columns=["appid", "lowest_other_store_usd", "cheapshark_matches"])
raw = steamspy_raw.merge(store_raw, on="appid", how="left", validate="one_to_one").merge(cheap_raw, on="appid", how="left", validate="one_to_one")
raw["retrieved_at_utc"] = retrieved_at_utc
print("Records after joining:", len(raw), "| Steam Store matches:", len(store_raw), "| CheapShark responses:", len(cheap_raw))
display(raw.head())

Skipped cheapshark_108600.json: request remained unavailable
Skipped cheapshark_1966720.json: request remained unavailable
Skipped cheapshark_1174180.json: request remained unavailable
Unable to retrieve cheapshark_1517290.json: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Skipped cheapshark_1240440.json: request remained unavailable
Skipped cheapshark_648800.json: request remained unavailable
Skipped cheapshark_386360.json: request remained unavailable
Skipped cheapshark_444200.json: request remained unavailable
Skipped cheapshark_10.json: request remained unavailable
Records after joining: 100 | Steam Store matches: 100 | CheapShark responses: 91


,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu,store_name,release_date_raw,coming_soon,genre_primary,store_price_usd,store_list_price_usd,metacritic_score,lowest_other_store_usd,cheapshark_matches,retrieved_at_utc
0,730,Counter-Strike: Global Offensive,Valve,Valve,,7642084,1173003,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,1013936,Counter-Strike 2,"Aug 21, 2012",False,Action,0.00,0.00,NaN,NaN,0.0,2026-10-26T15:55:42.548164+00:00
1,1172470,Apex Legends,Respawn,Electronic Arts,,668053,326926,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,124262,Apex Legends™,"Nov 4, 2020",False,Action,0.00,0.00,88.0,NaN,0.0,2026-10-26T15:55:42.548164+00:00
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",,1520457,1037487,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,314682,PUBG: BATTLEGROUNDS,"Dec 21, 2017",False,Action,0.00,0.00,NaN,NaN,0.0,2026-10-26T15:55:42.548164+00:00
3,1623730,Palworld,Pocketpair,Pocketpair,,358266,22443,0,"50,000,000 .. 100,000,000",0,0,0,0,2999,2999,0,18028,Palworld,"Jul 9, 2026",False,Action,29.99,29.99,NaN,26.99,1.0,2026-10-26T15:55:42.548164+00:00
4,440,Team Fortress 2,Valve,Valve,,1044264,117208,0,"50,000,000 .. 100,000,000",0,0,0,0,0,0,0,43819,Team Fortress 2,"Oct 10, 2007",False,Action,0.00,0.00,92.0,NaN,0.0,2026-10-26T15:55:42.548164+00:00


## Resources and References
*What resources and references have you used for this project?*
📝 <!-- Answer Below -->

In [6]:
# ⚠️ Make sure you run this cell at the end of your notebook before every submission!
!jupyter nbconvert --to python source.ipynb

'c:\Users\Josh\AppData\Roaming\Python\Python314\Scripts\jupyter.exe' was blocked by your organization's Device Guard policy.
Contact your support person for more info.
